In [67]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import os
sns.set_theme()
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
cwd = os.getcwd()
print("Current working directory:", cwd)

# Change the working directory
#os.chdir('S:/OneDrive - University of Georgia/1 UGA/1 PhD/1 Spring24/Tobacco')
#os.chdir(r'/Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/1 Spring24/Tobacco')
os.chdir(r'/Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/Fall25/First_Chapter/madeData')

# Verify the change
print("New working directory:", os.getcwd())

Current working directory: /Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/Fall25/First_chapter/madeData
New working directory: /Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/Fall25/First_chapter/madeData


In [68]:
# Loading the Smoke and Heat dataset

df1 = pd.read_csv('/Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/1 Spring24/Tobacco/smoke_market1.csv')
smoke_market1_df_shape = df1.shape
df1['type'] = 1
df2 = pd.read_csv('/Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/1 Spring24/Tobacco/heat_market1.csv') 
heat_market1_df_shape = df2.shape
df2['type'] = 2




# Analyse the Smoke dataset

Rows and Columns

In [69]:

total_rows = len(df1)
print(f"Total number of rows: {total_rows}")

# List the column names
print(df1.columns)

Total number of rows: 1783225
Index(['sku', 'sku_id', 'brand_n', 'multi', 'size', 'W', 'totrev', 'qt',
       'price', 'region', 'type'],
      dtype='object')


Unique identifier

In [70]:
unique_counts = df1.nunique()
uniqueness_ratio = (unique_counts / total_rows) * 100

# Print columns that have 100% uniqueness
potential_ids = uniqueness_ratio[uniqueness_ratio == 100].index.tolist()

print("\n--- Uniqueness Report ---")
print(uniqueness_ratio.sort_values(ascending=False).to_string())

if potential_ids:
    print(f"\nPotential Unique Identifier(s) (100% Unique): **{potential_ids}**")
else:
    print("\nNo single column is 100% unique.")


--- Uniqueness Report ---
totrev     57.588190
qt          8.284148
price       0.252184
brand_n     0.059835
sku         0.059275
sku_id      0.059275
W           0.010262
region      0.000729
multi       0.000056
size        0.000056
type        0.000056

No single column is 100% unique.


In [71]:
# 1. Define the candidates
cols_to_check = ['brand_n', 'region', 'W']

# 2. Check for uniqueness
# Group by these 3 columns and count size
duplicate_check = df1.groupby(cols_to_check).size()

# 3. Print results
print(f"Checking combination: {cols_to_check}")
if duplicate_check.max() == 1:
    print("✅ Success! ['brand_n', 'region', 'W'] is the unique identifier.")
else:
    print(f"❌ Not unique. The maximum number of repeats for a combination is {duplicate_check.max()}.")
    
    # Optional: See which ones are duplicated
    print("\nSample duplicates:")
    print(df1[df1.duplicated(subset=cols_to_check, keep=False)].sort_values(by=cols_to_check).head())

Checking combination: ['brand_n', 'region', 'W']
✅ Success! ['brand_n', 'region', 'W'] is the unique identifier.


Variables with missing data

In [72]:
# Checking for missing variables
missing_counts = df1.isnull().sum()
print("--- Count of Missing Values Per Column ---")
print(missing_counts)

# Filter the series to only include counts > 0
missing_data_columns = missing_counts[missing_counts > 0].sort_values(ascending=False)

if missing_data_columns.empty:
    print("\n✅ Great news! No missing values found in your entire dataset.")
else:
    print("\n--- Columns with Missing Values (Count) ---")
    print(missing_data_columns.to_string())

--- Count of Missing Values Per Column ---
sku             0
sku_id          0
brand_n         0
multi           0
size            0
W               0
totrev     568059
qt         568059
price      568059
region          0
type            0
dtype: int64

--- Columns with Missing Values (Count) ---
totrev    568059
qt        568059
price     568059


Making the panel balanced

In [73]:
# 1. Get the unique count for each key variable
num_brands = df1['brand_n'].nunique()
num_regions = df1['region'].nunique()
num_weeks = df1['W'].nunique()

# 2. Calculate the expected number of rows for a balanced panel
expected_rows = num_brands * num_regions * num_weeks

print(f"Unique Brands: {num_brands}")
print(f"Unique Regions: {num_regions}")
print(f"Unique Weeks (W): {num_weeks}")
print(f"Expected Rows for a BALANCED panel: {expected_rows}")

actual_rows = len(df1)
print(f"Actual Rows in df1: {actual_rows}")

if actual_rows == expected_rows:
    print("\n✅ The panel is **BALANCED**. Every brand-region combination has data for every week.")
else:
    missing_rows = expected_rows - actual_rows
    print(f"\n❌ The panel is **UNBALANCED**. There are {missing_rows} missing brand-region-week combinations.")
    print("The actual number of rows is less than the expected number.")

Unique Brands: 1067
Unique Regions: 13
Unique Weeks (W): 183
Expected Rows for a BALANCED panel: 2538393
Actual Rows in df1: 1783225

❌ The panel is **UNBALANCED**. There are 755168 missing brand-region-week combinations.
The actual number of rows is less than the expected number.


In [74]:
import pandas as pd
from itertools import product

if actual_rows != expected_rows:
    
    # 1. Get all unique values for each key
    all_brands = df1['brand_n'].unique()
    all_regions = df1['region'].unique()
    all_weeks = df1['W'].unique()

    # 2. Create a DataFrame of ALL POSSIBLE combinations (the full panel structure)
    full_index = pd.DataFrame(
        product(all_brands, all_regions, all_weeks),
        columns=['brand_n', 'region', 'W']
    )

    # 3. Merge the full index with the actual data (using a left join)
    # The 'indicator=True' option tells us which rows only exist in the full_index
    merged_df = full_index.merge(
        df1,
        on=['brand_n', 'region', 'W'],
        how='left',
        indicator=True
    )

    # 4. Filter for rows that are missing in the original data ('left_only')
    missing_combinations = merged_df[merged_df['_merge'] == 'left_only'][['brand_n', 'region', 'W']]

    print("\n--- Missing Combinations Sample ---")
    if not missing_combinations.empty:
        # Display the first few missing combinations
        print(f"Total missing combinations found: {len(missing_combinations)}")
        print(missing_combinations.head())
    else:
        print("No missing combinations found (This should match the initial check).")


--- Missing Combinations Sample ---
Total missing combinations found: 755168
     brand_n  region  W
366        6       3  1
367        6       3  2
368        6       3  3
369        6       3  4
370        6       3  5


We donot have balanced panel, lets make one:

In [75]:

# 1. AGGREGATE DATA (With NaN protection)
# We use a lambda function to pass 'min_count=1'. 
# This ensures that if the original data was NaN, the sum remains NaN (instead of becoming 0).
df_aggregated = df1.groupby(['brand_n', 'region', 'W']).agg({
    'qt': lambda x: x.sum(min_count=1),      
    'totrev': lambda x: x.sum(min_count=1),  
    'price': 'mean', 
    'type': 'min'  
}).reset_index()

# 2. DEFINE THE SKELETON
unique_brands = df_aggregated['brand_n'].unique()
unique_regions = df_aggregated['region'].unique()
unique_weeks = df_aggregated['W'].unique()

full_index = pd.MultiIndex.from_product(
    [unique_brands, unique_regions, unique_weeks], 
    names=['brand_n', 'region', 'W']
)

# 3. REINDEX (Add the new missing rows)
# Original rows keep their data (including original NaNs preserved above).
# New rows are created as NaN.
df1_balanced = df_aggregated.set_index(['brand_n', 'region', 'W']).reindex(full_index).reset_index()

# 4. CLEANUP
cols_to_drop = ['sku', 'sku_id', 'multi', 'size']
df1_balanced = df1_balanced.drop(columns=cols_to_drop, errors='ignore')

# Fill only 'type' as requested
df1_balanced['type'] = df1_balanced['type'].fillna(1)

# --- VERIFICATION ---
print(f"Final Row Count: {len(df1_balanced)}")
print("\n--- Missing Value Counts (Includes both original and new NaNs) ---")
print(df1_balanced[['qt', 'totrev', 'price']].isnull().sum())

Final Row Count: 2538393

--- Missing Value Counts (Includes both original and new NaNs) ---
qt        1323227
totrev    1323227
price     1323227
dtype: int64


# Analyse the Smoke dataset
Rows and Columns

In [76]:
total_rows = len(df2)
print(f"Total number of rows: {total_rows}")

# List the column names
print(df2.columns)


Total number of rows: 374008
Index(['sku', 'sku_id', 'brand_n', 'multi', 'size', 'W', 'totrev', 'qt',
       'price', 'region', 'type'],
      dtype='object')


In [77]:
#Unique identifier
unique_counts = df2.nunique()
uniqueness_ratio = (unique_counts / total_rows) * 100

# Print columns that have 100% uniqueness
potential_ids = uniqueness_ratio[uniqueness_ratio == 100].index.tolist()

print("\n--- Uniqueness Report ---")
print(uniqueness_ratio.sort_values(ascending=False).to_string())

if potential_ids:
    print(f"\nPotential Unique Identifier(s) (100% Unique): **{potential_ids}**")
else:
    print("\nNo single column is 100% unique.")


--- Uniqueness Report ---
totrev     37.388772
qt         14.076704
price       0.303737
brand_n     0.068180
sku         0.065774
sku_id      0.065774
W           0.048929
region      0.003476
multi       0.000267
size        0.000267
type        0.000267

No single column is 100% unique.


In [78]:


# 1. Define the candidates
cols_to_check = ['brand_n', 'region', 'W']

# 2. Check for uniqueness
# Group by these 3 columns and count size
duplicate_check = df1.groupby(cols_to_check).size()

# 3. Print results
print(f"Checking combination: {cols_to_check}")
if duplicate_check.max() == 1:
    print("✅ Success! ['brand_n', 'region', 'W'] is the unique identifier.")
else:
    print(f"❌ Not unique. The maximum number of repeats for a combination is {duplicate_check.max()}.")
    
    # Optional: See which ones are duplicated
    print("\nSample duplicates:")
    print(df2[df2.duplicated(subset=cols_to_check, keep=False)].sort_values(by=cols_to_check).head())

Checking combination: ['brand_n', 'region', 'W']
✅ Success! ['brand_n', 'region', 'W'] is the unique identifier.


Variables with missing data

In [79]:
# Checking for missing variables
missing_counts = df2.isnull().sum()
print("--- Count of Missing Values Per Column ---")
print(missing_counts)

# Filter the series to only include counts > 0
missing_data_columns = missing_counts[missing_counts > 0].sort_values(ascending=False)

if missing_data_columns.empty:
    print("\n✅ Great news! No missing values found in your entire dataset.")
else:
    print("\n--- Columns with Missing Values (Count) ---")
    print(missing_data_columns.to_string())

--- Count of Missing Values Per Column ---
sku             0
sku_id          0
brand_n         0
multi           0
size            0
W               0
totrev     225197
qt         225197
price      225197
region          0
type            0
dtype: int64

--- Columns with Missing Values (Count) ---
totrev    225197
qt        225197
price     225197


Making the panel balanced

In [80]:
# 1. Get the unique count for each key variable
num_brands = df2['brand_n'].nunique()
num_regions = df2['region'].nunique()
num_weeks = df2['W'].nunique()

# 2. Calculate the expected number of rows for a balanced panel
expected_rows = num_brands * num_regions * num_weeks

print(f"Unique Brands: {num_brands}")
print(f"Unique Regions: {num_regions}")
print(f"Unique Weeks (W): {num_weeks}")
print(f"Expected Rows for a BALANCED panel: {expected_rows}")

actual_rows = len(df2)
print(f"Actual Rows in df2: {actual_rows}")

if actual_rows == expected_rows:
    print("\n✅ The panel is **BALANCED**. Every brand-region combination has data for every week.")
else:
    missing_rows = expected_rows - actual_rows
    print(f"\n❌ The panel is **UNBALANCED**. There are {missing_rows} missing brand-region-week combinations.")
    print("The actual number of rows is less than the expected number.")


if actual_rows != expected_rows:
    
    # 1. Get all unique values for each key
    all_brands = df2['brand_n'].unique()
    all_regions = df2['region'].unique()
    all_weeks = df2['W'].unique()

    # 2. Create a DataFrame of ALL POSSIBLE combinations (the full panel structure)
    full_index = pd.DataFrame(
        product(all_brands, all_regions, all_weeks),
        columns=['brand_n', 'region', 'W']
    )

    # 3. Merge the full index with the actual data (using a left join)
    # The 'indicator=True' option tells us which rows only exist in the full_index
    merged_df = full_index.merge(
        df2,
        on=['brand_n', 'region', 'W'],
        how='left',
        indicator=True
    )

    # 4. Filter for rows that are missing in the original data ('left_only')
    missing_combinations = merged_df[merged_df['_merge'] == 'left_only'][['brand_n', 'region', 'W']]

    print("\n--- Missing Combinations Sample ---")
    if not missing_combinations.empty:
        # Display the first few missing combinations
        print(f"Total missing combinations found: {len(missing_combinations)}")
        print(missing_combinations.head())
    else:
        print("No missing combinations found (This should match the initial check).")

Unique Brands: 255
Unique Regions: 13
Unique Weeks (W): 183
Expected Rows for a BALANCED panel: 606645
Actual Rows in df2: 374008

❌ The panel is **UNBALANCED**. There are 232637 missing brand-region-week combinations.
The actual number of rows is less than the expected number.

--- Missing Combinations Sample ---
Total missing combinations found: 232637
     brand_n  region  W
106       92       1  1
107       92       1  2
108       92       1  3
109       92       1  4
110       92       1  5


We donot have balanced panel, lets make one:

In [81]:


# 1. AGGREGATE DATA (With NaN protection)
# We use a lambda function to pass 'min_count=1'. 
# This ensures that if the original data was NaN, the sum remains NaN (instead of becoming 0).
df_aggregated = df2.groupby(['brand_n', 'region', 'W']).agg({
    'qt': lambda x: x.sum(min_count=1),      
    'totrev': lambda x: x.sum(min_count=1),  
    'price': 'mean', 
    'type': 'min'  
}).reset_index()

# 2. DEFINE THE SKELETON
unique_brands = df_aggregated['brand_n'].unique()
unique_regions = df_aggregated['region'].unique()
unique_weeks = df_aggregated['W'].unique()

full_index = pd.MultiIndex.from_product(
    [unique_brands, unique_regions, unique_weeks], 
    names=['brand_n', 'region', 'W']
)

# 3. REINDEX (Add the new missing rows)
# Original rows keep their data (including original NaNs preserved above).
# New rows are created as NaN.
df2_balanced = df_aggregated.set_index(['brand_n', 'region', 'W']).reindex(full_index).reset_index()

# 4. CLEANUP
cols_to_drop = ['sku', 'sku_id', 'multi', 'size']
df2_balanced = df2_balanced.drop(columns=cols_to_drop, errors='ignore')

# Fill only 'type' as requested
df2_balanced['type'] = df2_balanced['type'].fillna(2)

# --- VERIFICATION ---
print(f"Final Row Count: {len(df2_balanced)}")
print("\n--- Missing Value Counts (Includes both original and new NaNs) ---")
print(df2_balanced[['qt', 'totrev', 'price']].isnull().sum())

df2_balanced

Final Row Count: 606645

--- Missing Value Counts (Includes both original and new NaNs) ---
qt        457834
totrev    457834
price     457834
dtype: int64


,brand_n,region,W,qt,totrev,price,type
0,92,1,78,NaN,NaN,NaN,2.0
1,92,1,79,NaN,NaN,NaN,2.0
2,92,1,80,NaN,NaN,NaN,2.0
3,92,1,81,NaN,NaN,NaN,2.0
4,92,1,82,NaN,NaN,NaN,2.0
...,...,...,...,...,...,...,...
606640,1162,13,73,NaN,NaN,NaN,2.0
606641,1162,13,74,NaN,NaN,NaN,2.0
606642,1162,13,75,NaN,NaN,NaN,2.0
606643,1162,13,76,NaN,NaN,NaN,2.0


We have the balanced panel now with 2538393 rows with 1067 brands of Smoke tobacco and 606645 rows with 255 brands of Heated tobacco.
1323227 rows of Smoke and 457834 rows of Heat tobacco have missing qty, total revenue and price variables. This dataset included both regional and national data so have 13 regions (1-12 region & 13 national)

In [82]:
# Lets append these two dataset

In [83]:
import pandas as pd

def check_data_sufficiency(df, dataset_name):
    print(f"\n====== CHECKING: {dataset_name} ======")
    
    # 1. Group by Region and Week
    # We count non-null values. If count is 0, it means ALL brands are NaN for that week.
    coverage = df.groupby(['region', 'W'])[['price', 'qt', 'totrev']].count()
    
    # 2. Identify Region-Weeks with ZERO valid data
    # (i.e., The region existed, but every single brand had missing values)
    empty_price = coverage[coverage['price'] == 0]
    empty_qt = coverage[coverage['qt'] == 0]
    empty_rev = coverage[coverage['totrev'] == 0]
    
    # 3. Report Results
    total_rw = len(coverage)
    print(f"Total Region-Week combinations: {total_rw}")
    
    if empty_price.empty and empty_qt.empty and empty_rev.empty:
        print(f"✅ PASSED: All {total_rw} Region-Weeks have at least one valid observation.")
    else:
        print("❌ FAILED: Some Region-Weeks have NO valid data (all brands are NaN).")
        
        if not empty_price.empty:
            print(f"  - Missing Price in {len(empty_price)} Region-Weeks.")
            # Show first 5 missing
            print(f"    Sample: {empty_price.index.tolist()[:5]}")
            
        if not empty_qt.empty:
            print(f"  - Missing Quantity in {len(empty_qt)} Region-Weeks.")
            
        if not empty_rev.empty:
            print(f"  - Missing Revenue in {len(empty_rev)} Region-Weeks.")

# --- RUN THE CHECK ---

# 1. Check your balanced Smoke dataset
check_data_sufficiency(df1_balanced, "Smoke Dataset (df1)")

# 2. Check your Heated Tobacco dataset
# (Replace 'df2' with the actual name of your heated tobacco dataframe)
# If you haven't loaded it yet, ensure you load it before running this line.
try:
    check_data_sufficiency(df2_balanced, "Heated Tobacco Dataset (df2)") 
except NameError:
    print("\n⚠️ Note: 'df2' is not defined yet. Please load your Heated Tobacco dataset to check it.")


====== CHECKING: Smoke Dataset (df1) ======
Total Region-Week combinations: 2379
✅ PASSED: All 2379 Region-Weeks have at least one valid observation.

====== CHECKING: Heated Tobacco Dataset (df2) ======
Total Region-Week combinations: 2379
✅ PASSED: All 2379 Region-Weeks have at least one valid observation.


Lets Append these two dataset

In [84]:
df = pd.concat([df1_balanced, df2_balanced], ignore_index=True)
smoke_heat_df_shape = df.shape
df


,brand_n,region,W,qt,totrev,price,type
0,1,4,1,0.053,15.651,297.0,1.0
1,1,4,2,0.053,15.651,297.0,1.0
2,1,4,3,NaN,NaN,NaN,1.0
3,1,4,4,0.053,15.651,297.0,1.0
4,1,4,5,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...
3145033,1162,13,73,NaN,NaN,NaN,2.0
3145034,1162,13,74,NaN,NaN,NaN,2.0
3145035,1162,13,75,NaN,NaN,NaN,2.0
3145036,1162,13,76,NaN,NaN,NaN,2.0


In [85]:
df['year'] = pd.to_datetime(df['W'], unit='W', origin=pd.Timestamp(2017, 1 ,1)).dt.year
df

,brand_n,region,W,qt,totrev,price,type,year
0,1,4,1,0.053,15.651,297.0,1.0,2017
1,1,4,2,0.053,15.651,297.0,1.0,2017
2,1,4,3,NaN,NaN,NaN,1.0,2017
3,1,4,4,0.053,15.651,297.0,1.0,2017
4,1,4,5,NaN,NaN,NaN,1.0,2017
...,...,...,...,...,...,...,...,...
3145033,1162,13,73,NaN,NaN,NaN,2.0,2018
3145034,1162,13,74,NaN,NaN,NaN,2.0,2018
3145035,1162,13,75,NaN,NaN,NaN,2.0,2018
3145036,1162,13,76,NaN,NaN,NaN,2.0,2018


In [86]:
# Print column types of both DataFrames
print("Data types in df:")
print(df.dtypes)


# Convert 'year' column to integer in both DataFrames
df['year'] = df['year'].astype(int)


Data types in df:
brand_n      int64
region       int64
W            int64
qt         float64
totrev     float64
price      float64
type       float64
year         int32
dtype: object


In [87]:

## Saving the merged dataset



df.to_csv('heat_smoke_balanced_FINAL.csv', index=False) #brand, region, week- level

df

,brand_n,region,W,qt,totrev,price,type,year
0,1,4,1,0.053,15.651,297.0,1.0,2017
1,1,4,2,0.053,15.651,297.0,1.0,2017
2,1,4,3,NaN,NaN,NaN,1.0,2017
3,1,4,4,0.053,15.651,297.0,1.0,2017
4,1,4,5,NaN,NaN,NaN,1.0,2017
...,...,...,...,...,...,...,...,...
3145033,1162,13,73,NaN,NaN,NaN,2.0,2018
3145034,1162,13,74,NaN,NaN,NaN,2.0,2018
3145035,1162,13,75,NaN,NaN,NaN,2.0,2018
3145036,1162,13,76,NaN,NaN,NaN,2.0,2018


In [88]:
# Lets make this dataset a region-week dataset
#df['Expenditure'] = df.groupby(['region', 'W', 'type'])['totrev'].transform('sum') ##sum of expenditure for all the products in that entity by type
#df['quantity'] = df.groupby(['region', 'W', 'type'])['qt'].transform('sum') 

df


,brand_n,region,W,qt,totrev,price,type,year
0,1,4,1,0.053,15.651,297.0,1.0,2017
1,1,4,2,0.053,15.651,297.0,1.0,2017
2,1,4,3,NaN,NaN,NaN,1.0,2017
3,1,4,4,0.053,15.651,297.0,1.0,2017
4,1,4,5,NaN,NaN,NaN,1.0,2017
...,...,...,...,...,...,...,...,...
3145033,1162,13,73,NaN,NaN,NaN,2.0,2018
3145034,1162,13,74,NaN,NaN,NaN,2.0,2018
3145035,1162,13,75,NaN,NaN,NaN,2.0,2018
3145036,1162,13,76,NaN,NaN,NaN,2.0,2018


In [89]:
import pandas as pd

# 1. AGGREGATE FIRST (Critical Step)
# Collapse the brand-level data into Region-Week-Type level
# This ensures there is exactly ONE row per Type per Region-Week
df_agg = df.groupby(['W', 'region', 'year', 'type'], as_index=False)[['totrev', 'qt']].sum()

# 2. Split the aggregated data
cols_to_keep = ['W', 'region', 'year', 'totrev', 'qt']

df_smoke = df_agg.loc[df_agg['type'] == 1, cols_to_keep].copy()
df_heat  = df_agg.loc[df_agg['type'] == 2, cols_to_keep].copy()

# 3. Rename columns
value_cols = ['totrev', 'qt']
suffix_smoke = {col: f"{col}_Smoke" for col in value_cols}
suffix_heat  = {col: f"{col}_Heat" for col in value_cols}




df_smoke.rename(columns=suffix_smoke, inplace=True)
df_heat.rename(columns=suffix_heat, inplace=True)

# 4. Merge
# We include 'year' in the key just to be safe, though 'W' and 'region' is usually enough
df_wide = pd.merge(df_smoke, df_heat, on=['W', 'region', 'year'], how='inner')

# 5. Check results
df_wide

,W,region,year,totrev_Smoke,qt_Smoke,totrev_Heat,qt_Heat
0,1,1,2017,2.574840e+06,5578.204,2.094940e+05,414.742
1,1,2,2017,2.729862e+06,5567.702,1.504587e+05,296.045
2,1,3,2017,1.046158e+06,2116.976,5.793492e+04,112.455
3,1,4,2017,3.198163e+06,7037.711,2.132549e+05,439.983
4,1,5,2017,1.337419e+07,29068.790,1.035635e+06,2102.634
...,...,...,...,...,...,...,...
2374,183,9,2020,1.348564e+06,2465.845,3.710347e+05,680.921
2375,183,10,2020,1.347240e+06,2565.614,5.031467e+05,950.301
2376,183,11,2020,3.096018e+06,6137.084,1.330899e+06,2506.863
2377,183,12,2020,4.644789e+06,8551.305,2.057523e+06,3654.706


In [90]:
df_wide.to_csv('heat_smoke_market_FINAL.csv', index=False) # region, week- level
